# Ejemplo de creación de una red neuronal Convolucional pero con Convoluciones  (filtros) PRE-Entrenados

Primero cargamos los imports necesarios.

In [58]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import Callback

import urllib
import os
import zipfile
import random

from shutil import copyfile

Creamos lsa constantes del notebook

In [59]:
#WEIGHTS_URL  = "https://storage.googleapis.com/mledudatasets/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
#WEIGHTS_URL = "https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
#WEIGHTS_FILE = 'inception_v3,h5'
CATSDOGS_URL = 'https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip'
CATSDOGS_FILE = 'kagglecatsanddogs.zip'
DOWNLOAD_DIR = '/tmp'
TRAINING_SIZE = 0.9

EPOCHS = 20
VERBOSE = 'auto'

STOP_ACCURACY =  0.9

Preparamos los sets de entrenamiento y test

In [60]:
urllib.request.urlretrieve(CATSDOGS_URL, CATSDOGS_FILE)
# Descomprimo el zip.
zip_ref = zipfile.ZipFile(CATSDOGS_FILE, 'r')
zip_ref.extractall(DOWNLOAD_DIR)
zip_ref.close()

# Check si hemos descomprimido todo bien.

print(f'Gatos: {len(os.listdir(f'{DOWNLOAD_DIR}/PetImages/Cat/'))}')
print(f'Perros: {len(os.listdir(f'{DOWNLOAD_DIR}/PetImages/Dog/'))}')

# Creamos las distintas carpetas de train y test
try:
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/train') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/train')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/test') == False:    
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/test')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/train/dogs') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/train/dogs')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/train/cats') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/train/cats')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/test/dogs') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/test/dogs')
    if os.path.exists(f'{DOWNLOAD_DIR}/cats-v-dogs/test/cats') == False:
        os.mkdir(f'{DOWNLOAD_DIR}/cats-v-dogs/test/cats')
except OSError as e:
    print (f'Error creando las carpetas: {e}')
    exit(1)
    #pass

Gatos: 12501
Perros: 12501


In [61]:
# Hacemos el split de los datos en entreno y testing

def split_data(source_folder, training_folder, testing_folder, split_size):
    files = []
    # Creo una lista de ficheros con todos los ficheros 
    for filename in os.listdir(source_folder):
        # Check si tiene contenido
        if os.path.getsize(f'{source_folder}/{filename}') > 0:
            files.append(filename)
        else:
            print(f'Se ignora el fichero {filename} de tamano 0.')

    #Calculamos el número de ficheros de entrenamiento y de testing
    training_length = int(len(files) * split_size)
    test_length = int(len(files) - training_length)
    # reordenamos el conjunto de imágenes de forma aleatoria / desordenada para no tener siempre las mismas imágenes en los juegos
    lista_desordenada = random.sample(files, len(files))
    lista_trainning = lista_desordenada[0: training_length]
    lista_test = lista_desordenada[training_length:]

    #procedo a copiar las imágenes a las carpetas de entrenamiento.
    for filename in lista_trainning:
        origin = f'{source_folder}/{filename}'
        destination = f'{training_folder}/{filename}'
        copyfile(origin, destination)

    # Procedo a copiar las imágenes a las carpeta de testing
    for filename in lista_test:
        origin = f'{source_folder}/{filename}'
        destination = f'{testing_folder}/{filename}'
        copyfile(origin, destination)
    
split_data(f'{DOWNLOAD_DIR}/PetImages/Cat', f'{DOWNLOAD_DIR}/cats-v-dogs/train/cats', f'{DOWNLOAD_DIR}/cats-v-dogs/test/cats', TRAINING_SIZE)
split_data(f'{DOWNLOAD_DIR}/PetImages/Dog', f'{DOWNLOAD_DIR}/cats-v-dogs/train/dogs', f'{DOWNLOAD_DIR}/cats-v-dogs/test/dogs', TRAINING_SIZE)


Se ignora el fichero 666.jpg de tamano 0.
Se ignora el fichero 11702.jpg de tamano 0.


Adaptamos los juegos de test y de entrenamiento

In [62]:
# Train...
# Creo el DataGenerator. Esta clase nos generará las combinaciones a partir de las imágenes que le pasemos.
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 40,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    fill_mode = 'nearest'
)
# Genero las imágenes de entrenamiento
train_generator = train_datagen.flow_from_directory( 
    f'{DOWNLOAD_DIR}/cats-v-dogs/train',
    batch_size = 100,
    class_mode = 'binary',
    target_size = (150,150)
)

# Test...
# Creo el DataGenerator. Solo tenemos que reescalarlas.
test_datagen = ImageDataGenerator(
    rescale = 1./255,
)
# Genero las imágenes de testing
test_generator = test_datagen.flow_from_directory( 
    f'{DOWNLOAD_DIR}/cats-v-dogs/test',
    batch_size = 100,
    class_mode = 'binary',
    target_size = (150,150)
)

Found 24998 images belonging to 2 classes.
Found 10205 images belonging to 2 classes.


Cargamos el modelo pre-cargado.

In [63]:
"""
# Forma antigua, no se usa por la nueva versión de keras

urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_FILE)
pre_trained_model = InceptionV3( 
    input_shape = (150,150,3),
    include_top = False,
    weights = None
)

pre_trained_model.load_weights(WEIGHTS_FILE)
"""

#Probamos la forma nueva via Keras. RECUERDA que wights ya carga los pesos descargados en el modelo.
pre_trained_model = InceptionV3(
    input_shape = (150,150,3),
    include_top = False,
    weights = 'imagenet'
)

Imprimimos un resumen de todas las capas que dispone pre-cargadas

In [64]:
pre_trained_model.summary()

Model: "inception_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 150, 150,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_376 (Conv2D) │ (None, 74, 74,    │        864 │ input_layer_4[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 74, 74,    │         96 │ conv2d_376[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_376      │ (None, 74, 74,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_377 (Conv2D) │ (None, 72, 72,    │      9,216 │ activation_376[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 72, 72,    │         96 │ conv2d_377[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_377      │ (None, 72, 72,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_378 (Conv2D) │ (None, 72, 72,    │     18,432 │ activation_377[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 72, 72,    │        192 │ conv2d_378[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_378      │ (None, 72, 72,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 35, 35,    │          0 │ activation_378[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_379 (Conv2D) │ (None, 35, 35,    │      5,120 │ max_pooling2d_16… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 35, 35,    │        240 │ conv2d_379[0][0]  │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_379      │ (None, 35, 35,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_380 (Conv2D) │ (None, 33, 33,    │    138,240 │ activation_379[0… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 33, 33,    │        576 │ conv2d_380[0][0]  │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_380      │ (None, 33, 33,    │          0 │ batch_normalizat

 Total params: 21,802,784 (83.17 MB)

 Trainable params: 21,768,352 (83.04 MB)

 Non-trainable params: 34,432 (134.50 KB)

Indicamos al modelo que las capas NO han de ser entrenadas.

In [65]:
for layer in pre_trained_model.layers:
    layer.trainable = False

Obtengo la ultima capa que queremos usar del modelo pre-entrenada

In [66]:
last_layer = pre_trained_model.get_layer('mixed7')
#print(f'last_layer output shape:  {last_layer.compute_output_shape([150,150,3])}')
print(f'last_layer output shape:  {last_layer.output.shape}')
last_output = last_layer.output

last_layer output shape:  (None, 7, 7, 768)


A partir de aqui, defino  las capas que usara mi modelo

In [67]:
# Trnasformo la salida de las convoluciones y filtros  (deberían ser imgs 7x7)  a una serie de pixeles.
x = layers.Flatten()(last_output)
# Añado una capa de 1024 neurona densas.
x = layers.Dense(1024, activation='relu')(x)
# Añado la capa sigmoide para la clasificacion binaria
x = layers.Dense(1, activation= 'sigmoid')(x)

Ahora defino mi modelo, incluyendo las capas que usaré.

In [68]:
model = Model(pre_trained_model.input, x)

Compilo el modelo...

In [69]:
model.compile(
    optimizer = RMSprop(learning_rate=0.0001),
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)
 

Llegados a aquí, ya tenemos el modelo compilado y procedemos a entrenar ( la capa de neuronas densas que hemos creado)

Pero antes generamos el Stopper callback

In [70]:
class Stopper(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        accuracy = logs.get('accuracy')
        if accuracy is None:
            accuracy = logs.get('acc')
        if accuracy is not None and accuracy > STOP_ACCURACY:
            print(f'\n Se ha superado la precisión deseada {accuracy}')
            self.model.stop_training = True


In [ ]:
# Entrenamos el modelo.
# https://github.com/lmoroney/tfbook/blob/master/chapter2/transfer-learning.py
# Llamamos al entrenamiento con los generators.

historia = model.fit(
    train_generator,
    validation_data = test_generator,
    epochs = EPOCHS,
    verbose = VERBOSE,
    callbacks = [Stopper()]
)

"""
historia = model.fit_generator(
    train_generator,
    validation_data = test_generator,
    epochs = EPOCHS,
    verbose = VERBOSE
)
"""

TypeError: TensorFlowTrainer.fit() got an unexpected keyword argument 'callback'

In [ ]:
# Fin del notebook: el historial de entrenamiento queda en `historia`.
